# 21장 · 문자열 청킹과 원문 오프셋

청킹은 긴 문서를 일정한 기준으로 잘라 검색 단위로 만드는 작업입니다. 겹침을 두면 경계 부근의 글자가 두 청크에 함께 들어갑니다.

도서관 안내에서 ‘14일입니다.’를 찾았는데 잘라 둔 조각이 ‘일입니다.’로 시작한다면 숫자는 어디로 갔을까요? 글을 자른 위치와 앞 조각에 겹친 부분을 원문과 나란히 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 문자열 슬라이싱과 chunk overlap

청크는 검색에 넣으려고 나눈 원문의 한 조각입니다. 이 실험에서는 파이썬 문자열의 문자 수를 기준으로 16자씩 자르고, 이웃 조각 사이에 4자를 겹칩니다. 모델 토큰 수로 자른 실험과는 단위가 다릅니다.

**슬라이싱과 청크 경계**　16자씩 자르고 4자를 겹치면 두 번째 조각은 인덱스 12에서 시작합니다.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter21/index.html#practice-space)


### 원문과 문서 ID 함께 읽기

<p>doc의 text에는 대출 기간 문장이 있고 id는 library-1입니다. 글자를 자르기 전 원문 ID를 기록해 두면 잘린 문장이 어느 문서에서 나왔는지 거슬러 올라갈 수 있습니다.</p>


### 셀 01 · 문자열 슬라이스의 끝 위치

20장 첫 문서의 온점이 첫 조각에 들어가는지 보세요.


In [1]:
doc = {"id":"library-1","text":"도서 대출 기간은 14일입니다.","source":"가상 도서관 이용 안내","page":1}
text = doc["text"]
first = text[:16]
print(first)

도서 대출 기간은 14일입니다


‘도서 대출 기간은 14일입니다’까지 들어가고 맨 끝 온점은 다음에 남습니다. 범위의 끝 번호 16은 포함하지 않습니다.


### 청킹 전 텍스트 정제

<p>문서를 나누는 작업을 청킹이라고 합니다. 청킹 전에 파싱 결과의 문단·페이지·숫자가 올바른지 확인해야 합니다. 잘못된 순서로 읽힌 표를 작은 조각으로 나누어도 원래 의미가 복구되지는 않습니다.</p>

<p>정리 규칙은 필요에 맞춰 최소한으로 적용합니다. 이 예제는 줄바꿈 형식을 통일하고 문서 앞뒤 공백만 제거합니다. 문장 안의 숫자·단위·공백을 무조건 지우지 않습니다. 원본과 정리본을 나누어 보관해야 어떤 변경이 있었는지 설명할 수 있습니다.</p>


### 셀 02 · 청크 간 이동 간격

시작점을 16에서 겹칠 4만큼 앞으로 당깁니다.

첫 조각은 text[0:16]으로 인덱스 0~15를 담습니다. 끝점 16은 포함하지 않습니다. 네 글자를 겹치면 둘째 조각은 16−4=12에서 시작합니다. 인덱스 16은 처음부터 세어 열일곱 번째 글자입니다.


In [2]:
second = text[16 - 4:]
print(second)

일입니다.


둘째 조각에는 ‘일입니다.’가 들어 있습니다. 서로 이어 붙일 때 이 네 글자가 두 번 나타날 수 있습니다. 입력 문장을 바꾸지 않고 시작 위치만 확인하면 이유를 볼 수 있습니다.


### 청크 크기와 겹침

<p>크기는 한 조각에 들어갈 범위이고 중첩은 앞뒤 조각이 함께 포함하는 범위입니다. 경계에서 문장이 끊기는 영향을 줄이려고 중첩을 둘 수 있지만 중복 저장과 검색 결과 중복도 늘어날 수 있습니다.</p>

<p>이번 코드는 Python 문자열의 문자 위치로 나눕니다. 토큰 기반 청킹이 아니므로 size=40을 모델 토큰 40개라고 읽지 않습니다. 문장 경계를 인식하지 않는 단순 기준선을 먼저 관찰합니다.</p>

<table><caption>예제 설정</caption><thead><tr><th scope="col">항목</th><th scope="col">값</th><th scope="col">뜻</th></tr>
</thead><tbody><tr><td>size</td><td>16</td><td>한 청크 최대16문자</td></tr>
<tr><td>overlap</td><td>4</td><td>이전 청크의 끝4문자를 다음 시작에 포함</td></tr>
<tr><td>이동 간격</td><td>12</td><td>size-overlap</td></tr>
<tr><td>경계 기준</td><td>문자 위치</td><td>문장·토큰 의미를 검사하지 않음</td></tr>
</tbody></table>


### 과제 03 · 인접 슬라이스의 중복 구간

첫 조각 끝 4자와 둘째 조각 앞 4자가 같은지 검사하고 그 문자열을 출력하세요.


In [3]:
# 첫 조각의 끝과 둘째 조각의 앞을 비교하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>assert first[-4:] == second[:4]
print(first[-4:])</code></pre>
<p>실행 결과</p><pre><code>일입니다</code></pre>
<p>‘일입니다’가 두 조각에 같이 들어 있습니다. 이 결과는 온점 직전까지의 겹침을 설명합니다.</p>
</details>


### 나란한 청크의 겹친 문자열 확인

<p>first[-4:]와 second[:4]를 비교합니다. 두 값이 같은 ‘일입니다’이면 네 글자가 보존된 채 두 청크에 중복되어 있음을 확인할 수 있습니다. 두 값의 일치는 검색 품질까지 측정한 결과는 아닙니다.</p>


## 청크 ID와 시작·끝 위치

검색 결과에 조각 본문만 담으면 근거가 어느 자료 몇 쪽인지 잊기 쉽습니다. 표시할 문서에는 id, text, source, page를 두고 원문의 위치는 별도 표에 남길 수 있습니다.


### 청킹 매개변수 비교

<p>먼저 overlap=0에서 size=16과 size=24를 비교해 크기 효과를 봅니다. 다음에는 size=16을 고정하고 overlap=0과 overlap=4를 비교합니다. 표의 첫째·셋째 행이 중첩 비교 쌍입니다.</p>

<p>많은 청크가 생겼다는 사실만으로 검색이 좋아진 것은 아닙니다. 문서 하나를 답변에 필요한 정보가 나뉘도록 잘랐는지, 같은 문장이 반복되는지, 질문에 필요한 단위가 함께 남아 있는지를 직접 읽어야 합니다.</p>

<table><caption>비교 기록 양식</caption><thead><tr><th scope="col">크기</th><th scope="col">중첩</th><th scope="col">청크 수</th><th scope="col">중복 문자 수</th><th scope="col">관찰할 내용</th></tr>
</thead><tbody><tr><td>16</td><td>0</td><td>직접 기록</td><td>직접 기록</td><td>경계에서 끊긴 문장</td></tr>
<tr><td>24</td><td>0</td><td>직접 기록</td><td>직접 기록</td><td>한 조각에 남은 정보</td></tr>
<tr><td>16</td><td>4</td><td>직접 기록</td><td>직접 기록</td><td>앞 조각 끝과 다음 시작의 반복</td></tr>
</tbody></table>

<p>짧은 기준 문서는 size=24로 나누면 모두 한 조각 안에 들어갑니다. 이때 overlap만 바꿔도 실제로 반복되는 경계가 없을 수 있습니다. 중첩을 관찰할 때는 두 조각 이상 생기는 size=16 조건을 사용합니다. 청크 수가 같아도 포함된 글자의 중복은 다를 수 있습니다.</p>


### 셀 04 · 청크의 원문 오프셋

실험용 식별자에는 0부터 센 조각 번호를 붙입니다.


In [4]:
chunk = {"id": f"{doc['id']}:1", "text": second, "source": doc["source"], "page": doc["page"]}
location = {"document_id": doc["id"], "start": 12, "end": len(text)}
print(chunk["id"], text[location["start"]:location["end"]])

library-1:1 일입니다.


library-1:1과 ‘일입니다.’가 나옵니다. ID만 읽어 위치를 추측하지 않고 기록한 범위에서 본문을 되찾았습니다.


### 청크의 시작·끝 위치로 원문 복원

<p>location['start']는 12, end는 원문 길이입니다. text[start:end]로 자른 값이 둘째 청크인 ‘일입니다.’와 같은지 확인해 보세요. end가 가리키는 위치는 결과에 포함되지 않습니다.</p>


### 과제 05 · overlap 0의 청크 경계

크기는 16으로 유지하고 중첩만 0으로 바꾸세요. 둘째 조각에 무엇이 남나요?


In [5]:
# 같은 원문의 16번째 위치부터 잘라 보세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(text[16:])</code></pre>
<p>실행 결과</p><pre><code>.</code></pre>
<p>온점 한 글자가 남습니다. 조각을 더 많이 만들었다고 검색에 쓸 근거가 항상 더 좋아지는 것은 아닙니다.</p>
</details>


### 원문 끝에서 짧아지는 조각

<p>원문을 인덱스 16부터 잘라 보면 온점 한 글자가 남습니다. 겹침을 4로 주면 둘째 조각이 12부터 시작해 ‘일입니다.’를 담습니다. 같은 마지막 글자가 두 설정에서 어떤 주변 문맥을 얻는지 비교해 보세요.</p>


## 청킹 설정과 다음 단계의 입력

다음 장에서는 세 자료에 각각 하나의 벡터를 연결하려고 공통 기준 자료를 씁니다. 이 기준에서는 한 쪽 전체가 한 조각이며 ID는 library-1 그대로입니다. 방금 만든 library-1:1은 크기 비교용으로만 보관합니다.


### 셀 06 · 청크 ID와 원문 ID

기준 문서 한 건에서 출처가 있는 본문을 통째로 읽습니다.


In [6]:
print(doc["id"], doc["text"])

library-1 도서 대출 기간은 14일입니다.


library-1의 전체 대출 문장이 온점까지 남습니다. 청킹 방식을 바꾼 실험을 정식 입력으로 채택하려면 원문과 청크를 대응하는 표와 이후 벡터도 함께 새로 만들어야 합니다.


### 원문 ID와 청크 ID 구분

<p>library-1은 문서 전체, library-1:1은 그 문서에서 잘라낸 조각의 ID입니다. 다음 실험에 넘길 때 원문 ID와 시작·끝 인덱스를 함께 보존하면 조각의 문장을 원문에서 다시 찾을 수 있습니다.</p>


## 돌아보기

원문을 합칠 때는 겹친 구간을 한 번만 세고, 출처를 표시할 때는 원문 ID와 조각 ID 중 어떤 단위를 쓰는지 알려 주세요. 눈으로 차이를 구분한 뒤 size와 overlap을 한 번에 하나만 바꿔 보세요.
